In [ ]:
# !pip install ctgan sdv pandas numpy

import pandas as pd
import numpy as np
from ctgan import CTGAN

# ---------------------------------------------------------
# 1. LOAD DỮ LIỆU & CẤU HÌNH
# ---------------------------------------------------------
print("Đang đọc dữ liệu...")
data = pd.read_csv('Agri_Data_Cleaned.csv')

discrete_columns = [
    'District', 'Season', 'Crop Name', 'Transplant', 
    'Growth', 'Harvest', 'pH_Suitability', 
    'Dominant_Soil_Texture', 'Water_Availability_Cat',
    'Extreme_Heat_Risk', 'Is_Extreme_Heat',
    'is_extreme_Heat_Stress_Days', 
    'is_extreme_Wind_Max'
]

# ---------------------------------------------------------
# 2. HUẤN LUYỆN CTGAN
# ---------------------------------------------------------
print("Đang khởi tạo và huấn luyện CTGAN (Epochs=300)...")
ctgan = CTGAN(epochs=300, batch_size=500, verbose=True)
ctgan.fit(data, discrete_columns)

# ---------------------------------------------------------
# 3. SINH DỮ LIỆU
# ---------------------------------------------------------
print("Đang sinh dữ liệu mới...")
num_rows = 20000
synthetic_data = ctgan.sample(num_rows)

print("Đang thực hiện Hậu kiểm Logic (Logic Enforcement)...")

# =========================================================
# 4. HẬU XỬ LÝ TOÀN DIỆN (COMPREHENSIVE POST-PROCESSING)
# =========================================================

# --- A. ĐỒNG BỘ CÂY TRỒNG & THỜI GIAN (NATURAL SEASONALITY) ---
# Sử dụng lấy mẫu có trọng số (Weighted Sampling) để giữ độ đa dạng tự nhiên
cols_to_sync = ['Season', 'Transplant', 'Growth', 'Harvest']
synthetic_data = synthetic_data.drop(columns=cols_to_sync, errors='ignore')

sampled_rows = []
for crop in synthetic_data['Crop Name'].unique():
    idx = synthetic_data[synthetic_data['Crop Name'] == crop].index
    count = len(idx)
    
    orig_subset = data[data['Crop Name'] == crop][cols_to_sync]
    
    if not orig_subset.empty:
        # Lấy mẫu ngẫu nhiên (có hoàn lại) đúng phân phối thực tế
        sampled = orig_subset.sample(n=count, replace=True)
        sampled.index = idx
    else:
        sampled = pd.DataFrame(np.nan, index=idx, columns=cols_to_sync)
        
    sampled_rows.append(sampled)

sampled_df = pd.concat(sampled_rows)
synthetic_data = pd.concat([synthetic_data, sampled_df], axis=1)


# --- B. ĐẶC THÙ ĐỊA LÝ & KHÍ HẬU (GEO-CLIMATE BOUNDARIES) ---
# Khóa lượng mưa theo đặc thù từng Huyện và Mùa
geo_climate = data.groupby(['District', 'Season'])['Rainfall'].agg(Rain_min='min', Rain_max='max').reset_index()
dist_climate = data.groupby('District')['Rainfall'].agg(Dist_min='min', Dist_max='max').reset_index()

synthetic_data = synthetic_data.merge(geo_climate, on=['District', 'Season'], how='left')
synthetic_data = synthetic_data.merge(dist_climate, on='District', how='left')

# Cơ chế Fallback 3 lớp chặn NaN
global_min = data['Rainfall'].min()
global_max = data['Rainfall'].max()

synthetic_data['Rain_min'] = synthetic_data['Rain_min'].fillna(synthetic_data['Dist_min']).fillna(global_min)
synthetic_data['Rain_max'] = synthetic_data['Rain_max'].fillna(synthetic_data['Dist_max']).fillna(global_max)

synthetic_data['Rainfall'] = synthetic_data['Rainfall'].clip(
    lower=synthetic_data['Rain_min'], 
    upper=synthetic_data['Rain_max']
).round(2)

synthetic_data = synthetic_data.drop(columns=['Rain_min', 'Rain_max', 'Dist_min', 'Dist_max'])


# --- HELPER: Hàm an toàn chia cho 0 ---
def safe_div_robust(numerator, denominator, epsilon=1e-6):
    denom_safe = denominator.copy()
    mask_too_small = np.abs(denom_safe) < epsilon
    denom_safe[mask_too_small] = np.sign(denom_safe[mask_too_small]) * epsilon
    denom_safe[denom_safe == 0] = epsilon
    return numerator / denom_safe

# --- C. CHẶN BIÊN ÂM & LOGIC CƠ BẢN ---
non_negative_cols = [
    'Rainfall', 'Soil_Moisture_mm', 'Avg_Salinity_Index', 
    'Organic_Carbon', 'Nitrogen', 'sm_surface', 'sm_rootzone',
    'Wind_Mean', 'Wind_Max', 'Heat_Stress_Days',
    'EVI', 'LAI', 'FPAR' 
]
for col in non_negative_cols:
    if col in synthetic_data.columns:
        synthetic_data[col] = synthetic_data[col].clip(lower=0)

def enforce_min_mean_max(df, col_min, col_mean, col_max):
    if {col_min, col_mean, col_max}.issubset(df.columns):
        mask_wrong = df[col_min] > df[col_max]
        cols = [col_min, col_max]
        df.loc[mask_wrong, cols] = df.loc[mask_wrong, cols].values[:, ::-1]
        df[col_mean] = df[col_mean].clip(lower=df[col_min], upper=df[col_max])
    return df

synthetic_data = enforce_min_mean_max(synthetic_data, 'Min Temp', 'Avg Temp', 'Max Temp')
synthetic_data = enforce_min_mean_max(synthetic_data, 'Min Relative Humidity', 'Avg Humidity', 'Max Relative Humidity')
synthetic_data = enforce_min_mean_max(synthetic_data, 'NDVI_Season_Min', 'NDVI_Season_Mean', 'NDVI_Season_Max')

for col in ['Min Relative Humidity', 'Avg Humidity', 'Max Relative Humidity']:
    if col in synthetic_data.columns:
        synthetic_data[col] = synthetic_data[col].clip(0, 100).round(1)

# --- D. CHUẨN HÓA CẢM BIẾN & TỶ LỆ ---
weather_cols = ['Min Temp', 'Avg Temp', 'Max Temp']
if all(col in synthetic_data.columns for col in weather_cols):
    synthetic_data[weather_cols] = synthetic_data[weather_cols].round(1)

ndvi_cols = [c for c in synthetic_data.columns if 'NDVI' in c]
for col in ndvi_cols:
    synthetic_data[col] = synthetic_data[col].clip(-1.0, 1.0)

if 'Rain_Temp_Ratio' in synthetic_data.columns:
    temp_safe = synthetic_data['Avg Temp'].copy()
    ratio = safe_div_robust(synthetic_data['Rainfall'], temp_safe)
    ratio[temp_safe <= 0] = 0 
    synthetic_data['Rain_Temp_Ratio'] = ratio 

if 'CN_Ratio' in synthetic_data.columns:
    synthetic_data['Nitrogen'] = synthetic_data['Nitrogen'].clip(lower=0.001)
    synthetic_data['CN_Ratio'] = (synthetic_data['Organic_Carbon'] / synthetic_data['Nitrogen'])

if 'Moisture_Ratio' in synthetic_data.columns:
    synthetic_data['sm_surface'] = synthetic_data['sm_surface'].clip(lower=0.001)
    synthetic_data['Moisture_Ratio'] = (synthetic_data['sm_rootzone'] / synthetic_data['sm_surface'])


# --- E. LOGIC CỜ & RỦI RO (FLAG CONSISTENCY) ---
# 1. Heat Stress
THRESHOLD_HEAT_DAYS = 43.0 
if 'is_extreme_Heat_Stress_Days' in synthetic_data.columns and 'Heat_Stress_Days' in synthetic_data.columns:
    # Nếu cờ báo Cực đoan (1) -> Ép ngày > 43
    mask_extreme = synthetic_data['is_extreme_Heat_Stress_Days'] == 1
    mask_update = mask_extreme & (synthetic_data['Heat_Stress_Days'] < THRESHOLD_HEAT_DAYS)
    synthetic_data.loc[mask_update, 'Heat_Stress_Days'] = np.random.uniform(THRESHOLD_HEAT_DAYS, 65.0, size=mask_update.sum())
    
    # Nếu cờ báo Bình thường (0) -> Ép ngày < 43
    mask_normal = synthetic_data['is_extreme_Heat_Stress_Days'] == 0
    mask_update_normal = mask_normal & (synthetic_data['Heat_Stress_Days'] >= THRESHOLD_HEAT_DAYS)
    synthetic_data.loc[mask_update_normal, 'Heat_Stress_Days'] = np.random.uniform(0.0, 42.4, size=mask_update_normal.sum())

    synthetic_data['Heat_Stress_Days'] = (synthetic_data['Heat_Stress_Days'] * 2).round(0) / 2

# 2. Wind Logic
THRESHOLD_WIND = 11.0
if 'is_extreme_Wind_Max' in synthetic_data.columns and 'Wind_Max' in synthetic_data.columns:
    mask_extreme_wind = synthetic_data['is_extreme_Wind_Max'] == 1
    mask_update = mask_extreme_wind & (synthetic_data['Wind_Max'] < THRESHOLD_WIND)
    synthetic_data.loc[mask_update, 'Wind_Max'] = np.random.uniform(THRESHOLD_WIND, 35.0, size=mask_update.sum())
    
    mask_normal_wind = synthetic_data['is_extreme_Wind_Max'] == 0
    mask_update_normal = mask_normal_wind & (synthetic_data['Wind_Max'] >= THRESHOLD_WIND)
    synthetic_data.loc[mask_update_normal, 'Wind_Max'] = np.random.uniform(2.0, 10.0, size=mask_update_normal.sum())
    
    if 'Wind_Mean' in synthetic_data.columns:
         mask_wrong_wind = synthetic_data['Wind_Mean'] > synthetic_data['Wind_Max']
         synthetic_data.loc[mask_wrong_wind, 'Wind_Mean'] = synthetic_data.loc[mask_wrong_wind, 'Wind_Max'] * 0.8
    
    synthetic_data['Wind_Max'] = synthetic_data['Wind_Max'].round(2)
    if 'Wind_Mean' in synthetic_data.columns:
        synthetic_data['Wind_Mean'] = synthetic_data['Wind_Mean'].round(2)

# 3. Label-Flag Correlation
if 'Extreme_Heat_Risk' in synthetic_data.columns and 'Is_Extreme_Heat' in synthetic_data.columns:
    mask_contradiction = (synthetic_data['Is_Extreme_Heat'] == 1) & (synthetic_data['Extreme_Heat_Risk'] == 'Low Risk')
    if mask_contradiction.any():
        synthetic_data.loc[mask_contradiction, 'Extreme_Heat_Risk'] = 'High Risk' 

    mask_low_risk = synthetic_data['Extreme_Heat_Risk'] == 'Low Risk'
    synthetic_data.loc[mask_low_risk, 'Is_Extreme_Heat'] = 0


# --- F. XỬ LÝ NĂNG SUẤT & ĐẤT (YIELD & SOIL) ---
synthetic_data['Area'] = synthetic_data['Area'].clip(lower=1).round(0).astype(int)
synthetic_data['Yield'] = synthetic_data['Yield'].clip(lower=0)
synthetic_data['Production'] = (synthetic_data['Area'] * synthetic_data['Yield']).round(0).astype(int)
synthetic_data['Yield'] = synthetic_data['Production'] / synthetic_data['Area'] 

soil_cols = ['Sand', 'Silt', 'Clay']
if all(col in synthetic_data.columns for col in soil_cols):
    synthetic_data[soil_cols] = synthetic_data[soil_cols].clip(lower=0)
    total_soil = synthetic_data[soil_cols].sum(axis=1)
    mask_zero_sum = total_soil == 0
    if mask_zero_sum.any():
        synthetic_data.loc[mask_zero_sum, ['Sand', 'Silt', 'Clay']] = [33.33, 33.33, 33.34]
        total_soil[mask_zero_sum] = 100.0
        
    for col in soil_cols:
        synthetic_data[col] = (synthetic_data[col] / total_soil * 100)
        
    synthetic_data['Sand'] = synthetic_data['Sand'].round(2)
    synthetic_data['Silt'] = synthetic_data['Silt'].round(2)
    synthetic_data['Clay'] = (100.00 - synthetic_data['Sand'] - synthetic_data['Silt']).round(2)
    
    # Cân bằng thông minh (Smart Soil Balancer)
    mask_neg_clay = synthetic_data['Clay'] < 0
    mask_sand_dest = mask_neg_clay & (synthetic_data['Sand'] >= synthetic_data['Silt'])
    if mask_sand_dest.any():
        synthetic_data.loc[mask_sand_dest, 'Sand'] += synthetic_data.loc[mask_sand_dest, 'Clay']
    
    mask_silt_dest = mask_neg_clay & (synthetic_data['Sand'] < synthetic_data['Silt'])
    if mask_silt_dest.any():
        synthetic_data.loc[mask_silt_dest, 'Silt'] += synthetic_data.loc[mask_silt_dest, 'Clay']

    if mask_neg_clay.any():
        synthetic_data.loc[mask_neg_clay, 'Clay'] = 0

if 'pH' in synthetic_data.columns:
    synthetic_data['pH'] = synthetic_data['pH'].clip(3.5, 9.0).round(2)


# --- G. ĐỒNG BỘ NHÃN PHÂN LOẠI (CATEGORICAL SYNC) ---
# Bước quan trọng cuối cùng: Đảm bảo Nhãn (Chữ) khớp với Số liệu (Đã sửa)
print("Đang đồng bộ lại Nhãn phân loại...")

# 1. Đồng bộ Nhãn kết cấu đất
if 'Dominant_Soil_Texture' in synthetic_data.columns:
    conditions_soil = [
        (synthetic_data['Clay'] >= 40),
        (synthetic_data['Sand'] >= 50),
        (synthetic_data['Silt'] >= 50)
    ]
    choices_soil = ['Clayey', 'Sandy', 'Silty']
    synthetic_data['Dominant_Soil_Texture'] = np.select(conditions_soil, choices_soil, default='Loamy')

# 2. Đồng bộ Nhãn pH
if 'pH_Suitability' in synthetic_data.columns:
    conditions_ph = [
        (synthetic_data['pH'] < 5.5),
        (synthetic_data['pH'] > 7.5)
    ]
    choices_ph = ['Acidic', 'Alkaline']
    synthetic_data['pH_Suitability'] = np.select(conditions_ph, choices_ph, default='Optimal')


# ---------------------------------------------------------
# 5. LƯU FILE
# ---------------------------------------------------------
output_file = 'Agri_Data_CTGAN.csv'
synthetic_data.to_csv(output_file, index=False)
print(f"Hoàn tất! Dữ liệu đã được lưu tại: {output_file}")
print("Phiên bản Final: Đã kiểm duyệt toàn diện Logic Vật lý, Mùa vụ, Địa lý & Ngữ nghĩa.")

Đang đọc dữ liệu...
Đang khởi tạo và huấn luyện CTGAN (Epochs=300)...


Gen. (-01.32) | Discrim. (-00.31): 100%|██████████| 300/300 [07:42<00:00,  1.54s/it]


Đang sinh dữ liệu mới...
Đang thực hiện Hậu kiểm Logic (Logic Enforcement)...
Đang đồng bộ lại Nhãn phân loại...


C:\Users\AKANEMO\AppData\Local\Temp\ipykernel_25016\1468422558.py:113: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[20.03715404 25.35332513 25.06033839 23.04772291 23.20857171 24.65443978
 25.4365885  25.03143031 26.16603611 19.4313516  25.21057464 23.67320509
 23.94161128 23.23477559 25.31332218 19.69265621 25.17566845 24.4695775
 25.2685088  25.14776413 23.76977548 23.51061513 25.51701898 23.90868361
 20.15846114 25.8894537  23.62884347 27.8680604  23.40280344 28.67318712
 27.83349775 24.36006839 23.34026504 23.10312032 23.3017156  23.92756666
 23.10618001 24.25999119 25.00513599 25.1942333  26.11275972 23.62184703
 24.6344     16.68721716 25.45979085 25.9815094  22.74780894 24.57046817
 24.08114175 25.9853751  19.7714768  27.40148239 25.7777438  23.70911966
 24.94775261 23.51120325 19.97883624 24.07514122 26.84294637 23.19364014
 23.00852053 25.44310474 24.19984607 23.88255743 19.49952948 25.6425951

Hoàn tất! Dữ liệu đã được lưu tại: Agri_Data_CTGAN_20k_Flawless_Final.csv
Phiên bản Final: Đã kiểm duyệt toàn diện Logic Vật lý, Mùa vụ, Địa lý & Ngữ nghĩa.
